# M2 GPS track audit

The dataset guide defines `.track` rows as **longitude, latitude, timestamp**, with timestamps measured in **seconds**. This notebook audits every trip in both `Student folder/train` and `Student folder/student_test`.

In [1]:
import math
import os
from pathlib import Path

os.environ.setdefault('MPLCONFIGDIR', str(Path('.matplotlib').resolve()))
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import pandas as pd

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / 'Student folder').exists() and (PROJECT_ROOT.parent / 'Student folder').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
DATA_ROOT = PROJECT_ROOT / 'Student folder'
TABLES_DIR = PROJECT_ROOT / 'results' / 'tables'
FIGURES_DIR = PROJECT_ROOT / 'results' / 'figures'
TABLES_DIR.mkdir(parents=True, exist_ok=True)
FIGURES_DIR.mkdir(parents=True, exist_ok=True)
PARTITIONS = {
    'train': DATA_ROOT / 'train',
    'student_test': DATA_ROOT / 'student_test',
}
LARGE_GAP_SECONDS = 10.0
TRACK_COLUMNS = ['longitude', 'latitude', 'timestamp_seconds']

track_files = [(partition, path) for partition, root in PARTITIONS.items()
               for path in sorted(root.glob('*/*.track'))]
print(f'Found {len(track_files)} tracks: ' +
      ', '.join(f'{name}={sum(p == name for p, _ in track_files)}' for name in PARTITIONS))

Found 80 training tracks.


## Inspect one trip

Loads `00000001.track` as three whitespace-separated columns, without a header.

In [2]:
example_path = PARTITIONS['train'] / '00000001' / '00000001.track'
example = pd.read_csv(example_path, sep=r'\s+', header=None, names=TRACK_COLUMNS)
example.head(10)

,longitude,latitude,timestamp_seconds
0,8.595203,49.844053,1.0
1,8.595294,49.844441,2.0
2,8.595352,49.844883,3.0
3,8.595390,49.845379,4.0
4,8.595587,49.845750,5.0
5,8.595555,49.846167,6.0
6,8.595744,49.846565,7.0
7,8.595733,49.846931,8.0
8,8.595878,49.847374,9.0
9,8.595849,49.847684,10.0


In [3]:
def audit_track(track_path: Path, partition: str) -> tuple[dict, list[float]]:
    lines = track_path.read_text(encoding='utf-8', errors='replace').splitlines()
    nonblank = [line for line in lines if line.strip()]
    counters = dict(wrong_field_rows=0, missing_value_rows=0, nonnumeric_value_rows=0,
                    nonfinite_value_rows=0, longitude_out_of_range_rows=0,
                    latitude_out_of_range_rows=0, invalid_timestamp_rows=0)
    timestamps = []

    for line in nonblank:
        fields = line.split()
        if len(fields) != 3:
            counters['wrong_field_rows'] += 1
            counters['missing_value_rows'] += len(fields) < 3
            timestamps.append(None)
            continue
        values, has_nonnumeric, has_nonfinite = [], False, False
        for field in fields:
            try:
                value = float(field)
            except ValueError:
                has_nonnumeric, value = True, None
            if value is not None and not math.isfinite(value):
                has_nonfinite, value = True, None
            values.append(value)
        counters['nonnumeric_value_rows'] += has_nonnumeric
        counters['nonfinite_value_rows'] += has_nonfinite
        longitude, latitude, timestamp = values
        counters['longitude_out_of_range_rows'] += longitude is not None and not -180 <= longitude <= 180
        counters['latitude_out_of_range_rows'] += latitude is not None and not -90 <= latitude <= 90
        counters['invalid_timestamp_rows'] += timestamp is None
        timestamps.append(timestamp)

    gaps = [current - previous for previous, current in zip(timestamps, timestamps[1:])
            if previous is not None and current is not None]
    positive_gaps = [gap for gap in gaps if gap > 0]
    flags = []
    if any(counters.values()): flags.append('invalid_row_or_coordinate')
    if any(gap == 0 for gap in gaps): flags.append('duplicate_timestamp')
    if any(gap < 0 for gap in gaps): flags.append('timestamp_reversal')
    if any(gap > LARGE_GAP_SECONDS for gap in gaps): flags.append('large_gap')

    return ({'partition': partition, 'trip_id': track_path.stem, 'track_path': str(track_path),
             'total_lines': len(lines), 'nonblank_rows': len(nonblank), **counters,
             'valid_timestamp_pairs': len(gaps), 'zero_timestamp_diffs': sum(gap == 0 for gap in gaps),
             'negative_timestamp_diffs': sum(gap < 0 for gap in gaps),
             'median_positive_gap_seconds': pd.Series(positive_gaps).median() if positive_gaps else float('nan'),
             'max_positive_gap_seconds': max(positive_gaps, default=float('nan')),
             'large_gap_count': sum(gap > LARGE_GAP_SECONDS for gap in gaps),
             'flags': ';'.join(flags)}, positive_gaps)

## Audit all trips and retain the results in cells

In [4]:
audit_results = [audit_track(path, partition) for partition, path in track_files]
summary = pd.DataFrame(result for result, _ in audit_results).sort_values(['partition', 'trip_id']).reset_index(drop=True)
all_positive_gaps = [gap for _, gaps in audit_results for gap in gaps]
flagged_trips = summary.loc[summary['flags'].ne('')].copy()

display(summary)
display(flagged_trips)

,partition,trip_id,track_path,total_lines,nonblank_rows,wrong_field_rows,missing_value_rows,nonnumeric_value_rows,nonfinite_value_rows,longitude_out_of_range_rows,latitude_out_of_range_rows,valid_timestamp_pairs,zero_timestamp_diffs,negative_timestamp_diffs,median_positive_gap_seconds,max_positive_gap_seconds,large_gap_count,flags
0,train,00000001,Student folder\train\00000001\00000001.track,1315,1315,0,0,0,0,0,0,1314,0,0,1.0,16.0,1,large_gap
1,train,00000002,Student folder\train\00000002\00000002.track,7243,7243,0,0,0,0,0,0,7242,0,0,1.0,160.0,41,large_gap
2,train,00000003,Student folder\train\00000003\00000003.track,821,821,0,0,0,0,0,0,820,0,0,1.0,2.0,0,
3,train,00000004,Student folder\train\00000004\00000004.track,1064,1064,0,0,0,0,0,0,1063,0,0,1.0,51.0,215,large_gap
4,train,00000005,Student folder\train\00000005\00000005.track,1045,1045,0,0,0,0,0,0,1044,0,0,1.0,1.0,0,
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
75,train,00000095,Student folder\train\00000095\00000095.track,1551,1551,0,0,0,0,0,0,1550,0,0,1.0,2.0,0,
76,train,00000096,Student folder\train\00000096\00000096.track,3271,3271,0,0,0,0,0,0,3270,0,0,1.0,1.0,0,
77,train,00000097,Student folder\train\00000097\00000097.track,1740,1740,0,0,0,0,0,0,1739,0,0,1.0,2.0,0,
78,train,00000098,Student folder\train\00000098\00000098.track,5584,5584,0,0,0,0,0,0,5583,0,0,1.0,50.0,5,large_gap


,partition,trip_id,track_path,total_lines,nonblank_rows,wrong_field_rows,missing_value_rows,nonnumeric_value_rows,nonfinite_value_rows,longitude_out_of_range_rows,latitude_out_of_range_rows,valid_timestamp_pairs,zero_timestamp_diffs,negative_timestamp_diffs,median_positive_gap_seconds,max_positive_gap_seconds,large_gap_count,flags
0,train,00000001,Student folder\train\00000001\00000001.track,1315,1315,0,0,0,0,0,0,1314,0,0,1.0,16.0,1,large_gap
1,train,00000002,Student folder\train\00000002\00000002.track,7243,7243,0,0,0,0,0,0,7242,0,0,1.0,160.0,41,large_gap
3,train,00000004,Student folder\train\00000004\00000004.track,1064,1064,0,0,0,0,0,0,1063,0,0,1.0,51.0,215,large_gap
5,train,00000006,Student folder\train\00000006\00000006.track,1450,1450,0,0,0,0,0,0,1449,0,0,1.0,301.0,5,large_gap
8,train,00000011,Student folder\train\00000011\00000011.track,2828,2828,0,0,0,0,0,0,2827,0,0,1.0,909.0,1,large_gap
11,train,00000014,Student folder\train\00000014\00000014.track,448,448,0,0,0,0,0,0,447,0,0,1.0,72.0,5,large_gap
16,train,00000021,Student folder\train\00000021\00000021.track,15783,15783,0,0,0,0,0,0,15782,0,0,1.0,2161.0,2,large_gap
17,train,00000022,Student folder\train\00000022\00000022.track,474,474,0,0,0,0,0,0,473,0,0,1.0,22.0,9,large_gap
18,train,00000024,Student folder\train\00000024\00000024.track,801,801,0,0,0,0,0,0,800,0,0,1.0,458.0,2,large_gap
20,train,00000026,Student folder\train\00000026\00000026.track,361,361,0,0,0,0,0,0,360,0,0,1.0,17.0,23,large_gap


In [5]:
gaps = pd.Series(all_positive_gaps, name='positive_gap_seconds')
log_min, log_max = math.log10(gaps.min()), math.log10(gaps.max())
bins = [10 ** (log_min + i * (log_max - log_min) / 80) for i in range(81)]
plt.figure(figsize=(9, 5))
plt.hist(gaps, bins=bins, log=True, edgecolor='white')
plt.axvline(LARGE_GAP_SECONDS, color='crimson', linestyle='--', label='large-gap threshold (10 s)')
plt.xscale('log')
plt.xlabel('Positive timestamp gap (seconds)')
plt.ylabel('Adjacent pairs (log scale)')
plt.title('GPS timestamp gaps across training and student-test trips')
plt.legend()
plt.tight_layout()
plt.savefig(FIGURES_DIR / 'm2_sampling_gap_histogram.png', dpi=180)
plt.show()

print(f'Median: {gaps.median():.6g} s | p99: {gaps.quantile(0.99):.6g} s | max: {gaps.max():.6g} s')
print('Threshold: 10 seconds - ten times the nominal 1 Hz interval, well above the p99.')

Median: 1 s | p99: 3 s | max: 153064 s
Threshold: 10 seconds - ten times the nominal 1 Hz interval, well above the p99.


C:\Users\harih\AppData\Local\Temp\ipykernel_3980\733705493.py:12: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [6]:
summary.to_csv(TABLES_DIR / 'm2_per_trip_audit.csv', index=False)
flagged_trips.to_csv(TABLES_DIR / 'm2_flagged_trips.csv', index=False)
example.head(10).to_csv(TABLES_DIR / 'm2_00000001_preview.csv', index=False)
notes = [
    'Coordinate order: longitude, latitude, timestamp.',
    'Timestamp unit: seconds; nominal sampling rate: 1 Hz.',
    'Large-gap threshold: 10 seconds.',
    'Rationale: 10 seconds is ten times the nominal sampling interval and well above the observed p99.',
    f'Positive gaps: n={len(gaps)}, median={gaps.median():.6g} s, p99={gaps.quantile(0.99):.6g} s, max={gaps.max():.6g} s.',
    f'Trips audited: {len(summary)}; flagged trips: {len(flagged_trips)}.'
]
(TABLES_DIR / 'm2_audit_notes.txt').write_text('\n'.join(notes) + '\n', encoding='utf-8')
print(f'Wrote tables to {TABLES_DIR.resolve()} and the histogram to {FIGURES_DIR.resolve()}')

Wrote outputs to C:\Users\harih\OneDrive\Desktop\somesh\sem-5\ml\ML-Hackathon\audit_output_train
